# Day 4.3 — Deterministic Checks Before Model Judgment


## Before you begin

### Learning outcomes

- Prove some defects with a parser instead of paying a model to guess at them.
- Combine free deterministic evidence with one bounded model call.
- See a trace step that honestly reports zero tokens.

Architecture reference: [Day 4 diagrams D15](../../diagrams/source/day_04.md)

### Expected observation

The AST checker reports 3 findings in well under a millisecond for 0 tokens, and recall rises from 5/9 to 6/9 without a second model call.


## Concept briefing

## Deterministic tools before more model calls

Some findings need no model judgement at all. A Python parser can prove that a file calls
`eval`, that a function has a mutable default argument, and that an `except Exception:`
handler exists. Linters, tests, type checkers and security scanners give objective
evidence for the patterns they support, for free, in milliseconds, with the same answer
every time.

Model reviewers earn their cost on ambiguous intent, cross-cutting reasoning,
prioritisation and explanation. A strong system pairs deterministic evidence with bounded
judgement instead of paying three models to rediscover facts a parser already proved.

One consequence matters for measurement: a static checker has never heard of your answer
key, so it invents its own finding ids. Your evaluator therefore has to match a finding to
a defect by *location*, exactly as it must for a real model.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/review_team"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")


## Step 1 — Load the artifact


In [ ]:
# The artifact under review and the instructor's answer key.
# Both are plain files; nothing here is secret from you, but the answer key is never
# put into a reviewer's prompt.
ARTIFACT_PATH = PROJECT_ROOT / "data" / "seeded_artifact" / "order_service.py"
GOLDEN_PATH   = PROJECT_ROOT / "data" / "golden_defects.json"

SOURCE = ARTIFACT_PATH.read_text(encoding="utf-8")   # the shared, immutable artifact

print("Artifact file :", ARTIFACT_PATH.name)
print("Artifact lines:", len(SOURCE.splitlines()))
print("Answer key    :", GOLDEN_PATH.name)


## Step 2 — Some defects are facts, not opinions

Python can parse itself. `ast` turns the source into a tree, and walking that tree *proves* that the file calls `eval`, that a function has a mutable default argument, and that a bare `except Exception:` exists. No judgement is involved, so no model is needed.


In [ ]:
from review_team import deterministic_checks

checks = deterministic_checks(SOURCE)

print("Deterministic findings:", len(checks), "\n")
for finding in checks:
    print(f"line {finding.line:>3} | {finding.category:<15} | {finding.severity:<8} | "
          f"{finding.title}")
    print(f"         id       : {finding.id}")
    print(f"         evidence : {finding.evidence}")


## Step 3 — Notice the identifiers

The checker's ids start with `AST-`, not `DEF-`. That is deliberate: a real static analyser has never heard of our answer key. So the evaluator has to match a finding to a defect by **location** — same category, close enough line — exactly as it must for a real model whose ids are arbitrary strings.


In [ ]:
from review_team import match_to_golden
import json

golden = json.loads(GOLDEN_PATH.read_text(encoding="utf-8"))

for finding in checks:
    matched = match_to_golden(finding, golden)
    print(f"{finding.id:<26} -> golden defect {matched}")


## Step 4 — Measure what the check cost

This is the whole argument for running tools first: the same evidence, for nothing.


In [ ]:
from time import perf_counter

start = perf_counter()
for _ in range(100):
    deterministic_checks(SOURCE)          # run it 100 times so the timer has something to see
elapsed_ms = (perf_counter() - start) * 1000 / 100

print("Average time per AST pass: %.3f ms" % elapsed_ms)
print("Model calls used         : 0")
print("Tokens used              : 0")
print("Cost                     : $0.000000")
print("Result changes between runs: no (same tree, same answer, every time)")


## Step 5 — Checks plus one reviewer

`run_checks_plus_reviewer` runs the parser, then makes **one** model call, then merges both sets of findings. Watch the trace: the first step declares that it made no model call.


In [ ]:
# Build the reviewer. This is the ONLY place the notebook decides live vs mock.
from review_team import FallbackReviewer, MockStructuredReviewer, OpenRouterReviewer

SCENARIO = "blind_spots"          # which blind spots the scripted reviewer has

if LIVE:
    # FallbackReviewer tries the real model and, if the call fails for any reason,
    # prints one line and uses the mock for that call so the lesson never stops.
    provider = FallbackReviewer(OpenRouterReviewer(), MockStructuredReviewer(SCENARIO))
else:
    provider = MockStructuredReviewer(SCENARIO)

print("Reviewer provider:", type(provider).__name__)
print("Scenario         :", SCENARIO)


In [ ]:
from review_team import evaluate, run_checks_plus_reviewer, run_single_reviewer

single    = run_single_reviewer(SOURCE, provider)
augmented = run_checks_plus_reviewer(SOURCE, provider)

print("Trace of the augmented run:")
for step in augmented.trace:
    print(" ", step["step"])
    for key, value in step.items():
        if key != "step":
            print("     ", key, "=", value)


## Step 6 — Did it help, and what did it cost?

Compare the two rows. The AST pass added a defect the reviewer missed, and the model bill did not move at all.


In [ ]:
single_row    = evaluate(single, GOLDEN_PATH)
augmented_row = evaluate(augmented, GOLDEN_PATH)

for label, r in (("single_reviewer", single_row), ("checks_plus_reviewer", augmented_row)):
    print(f"{label:<22} found {r['found']}/9 | calls {r['model_calls']} | "
          f"tokens {r['tokens']} | merged duplicates {r['merged_duplicates']}")

print("\nDefect gained by adding the parser:",
      sorted(set(single_row["missed"]) - set(augmented_row["missed"])))
print("Extra model calls to gain it     :",
      augmented_row["model_calls"] - single_row["model_calls"])
print("Extra tokens to gain it          :",
      augmented_row["tokens"] - single_row["tokens"])


## Step 7 — Where the parser stops

The AST checker cannot tell you that a flat 20-unit discount can push a total negative, or that a quantity should never be negative. Those are business rules, not syntax. That is the boundary where model judgement starts to be worth paying for.


In [ ]:
print("Still missed after the parser ran:")
for defect_id in augmented_row["missed"]:
    item = next(x for x in golden if x["id"] == defect_id)
    print(f"   {defect_id}  line {item['line']:>3}  {item['title']}")
print("\nNone of these are syntax facts; each needs judgement about intent.")


### Try it yourself

The supervisor merged some findings in step 5. Predict: how many of the parser's 3 findings were things the reviewer had *already* reported, and how many were new?


In [ ]:
# --- Worked solution ---
# Merge the two groups by hand and ask the supervisor to report what it did.
from review_team import synthesize_with_report

reviewer_findings, _usage = provider.review(SOURCE, "general")
report = synthesize_with_report([checks, reviewer_findings])

print("Findings that arrived at the supervisor:", report.received)
print("Merged as duplicates                   :", report.merged_duplicates)
print("Kept in the final report               :", report.kept)
print()
new_count = len(checks) - report.merged_duplicates
print("So of the parser's %d findings, %d overlapped what the reviewer had already said"
      % (len(checks), report.merged_duplicates))
print("and %d was new (DEF-MNT-02, the broad exception handler)." % new_count)
print("Overlap is not waste here: the parser's version is *proof*, the reviewer's was a claim.")


### Checkpoint

**1. The parser and the reviewer both reported the `eval` call on line 20. Is that wasted work?**

<details><summary>Show answer</summary>

Not in this case — it is free (0 tokens) and it upgrades a model's claim into a parser's proof. It becomes waste when you pay a *second model call* to rediscover something ordinary code already established. That is the rule: prove what you can prove, then spend model calls on what is left.

</details>

**2. Why does the deterministic step write `"model_calls": 0` into the trace instead of simply leaving the field out?**

<details><summary>Show answer</summary>

Because an absent number gets silently filled in by whoever reads the table next. An explicit zero labelled `no model call` makes the free step visible in every cost comparison, and stops anyone attributing the parser's findings to the model's bill.

</details>

### Recap

- Limitation we saw: the single reviewer missed defects a parser can prove in a fraction of a millisecond.
- Layer we added: an AST checker whose findings carry their own ids, merged with the reviewer's by a bounded supervisor.
- Evidence it worked: recall 5/9 -> 6/9 with 0 extra model calls and 0 extra tokens; the trace shows the free step reporting zeros.
